# Remaining usable-time estimation
**Data are SYNTHETIC**, generated from documented assumptions (see `app/services/expiry_training.py`). High R² only shows the pipeline works; it is **not** evidence of real-world accuracy. This is an experimental estimate, **not** a food-safety certification.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import json, pandas as pd
from app.services import expiry_training as T
df = T.generate_synthetic_dataset(6000, 42)
print(df.shape); display(df.head()); print(df.isna().mean().round(3))

(6000, 6)


,food_category,storage_condition,packaging,temperature_c,elapsed_h,remaining_usable_h
0,cooked_meal,insulated_box,vacuum_sealed,33.6,2.25,4.022
1,bakery_dry,refrigerated,covered,8.4,19.79,52.210
2,ready_to_eat_snack,room_temp,sealed_container,25.8,1.79,4.770
3,soup_liquid,hot_holding,open,56.9,0.29,0.285
4,soup_liquid,room_temp,open,29.4,0.24,2.500


food_category         0.000
storage_condition     0.000
packaging             0.029
temperature_c         0.030
elapsed_h             0.000
remaining_usable_h    0.000
dtype: float64


## Assumption model
`life = base_life[category] * 2^((20-T)/10) * storage * packaging * noise`; `remaining = max(0, life - elapsed)`.

In [3]:
print(json.dumps({'BASE_LIFE_H': T.BASE_LIFE_H, 'STORAGE_FACTOR': T.STORAGE_FACTOR, 'PACKAGING_FACTOR': T.PACKAGING_FACTOR}, indent=2))

{
  "BASE_LIFE_H": {
    "cooked_meal": 6.0,
    "cooked_protein_meal": 4.0,
    "fresh_salad": 4.0,
    "soup_liquid": 6.0,
    "ready_to_eat_snack": 8.0,
    "bakery_dry": 36.0,
    "sweets_desserts": 24.0
  },
  "STORAGE_FACTOR": {
    "room_temp": 1.0,
    "refrigerated": 3.0,
    "frozen": 10.0,
    "hot_holding": 1.5,
    "insulated_box": 1.3
  },
  "PACKAGING_FACTOR": {
    "open": 0.8,
    "covered": 1.0,
    "sealed_container": 1.2,
    "vacuum_sealed": 1.5
  }
}


## Train, compare and save (Linear / Random Forest / Gradient Boosting; chosen by validation RMSE, test reported once)

In [4]:
meta = T.train_and_save(str(ROOT/'saved_models'/'usable_time'))
pd.DataFrame({m: {**{'val_'+k: v for k, v in r['validation'].items()}, **{'test_'+k: v for k, v in r['test'].items()}} for m, r in meta['results'].items()}).T.round(3)

,val_MAE_h,val_RMSE_h,val_R2,test_MAE_h,test_RMSE_h,test_R2
linear_regression,4.372,6.606,0.911,4.003,5.973,0.922
random_forest,2.180,4.407,0.960,1.832,3.641,0.971
gradient_boosting,2.513,4.395,0.961,2.171,3.737,0.970


In [5]:
print('Selected:', meta['selected_model'], '| 90% residual half-width (h):', round(meta['interval_halfwidth_h_q90'], 2))

Selected: gradient_boosting | 90% residual half-width (h): 7.34


In [6]:
from app.services.expiry_service import UsableTimeEstimator
est = UsableTimeEstimator.from_dir(ROOT/'saved_models'/'usable_time')
for storage in ['room_temp','refrigerated','hot_holding']:
    r = est.predict('cooked_meal', 90, storage)
    print(storage, r['estimated_remaining_minutes'], r['interval_minutes'], r['uncertainty_level'])

room_temp 177.9 [0.0, 618.4] high
refrigerated 3530.4 [3089.9, 3970.9] lower
hot_holding 0.0 [0.0, 440.5] high
